In [9]:
import pandas as pd
import requests


def get_usgs_sensor_data(site_id, start_dt, end_dt):
    """Fetch stage (00065) and discharge (00060) from USGS NWIS API.

    start_dt and end_dt should be ISO 8601 formatted strings or pd.Timestamp
    objects.
    """
    url = "https://waterservices.usgs.gov/nwis/iv/"

    start_iso = pd.to_datetime(start_dt).isoformat()
    end_iso = pd.to_datetime(end_dt).isoformat()

    params = {
        "format": "json",
        "sites": str(site_id).zfill(8),  # USGS site numbers are 8 digits
        "parameterCd": "00065,00060",
        "startDT": start_iso,
        "endDT": end_iso,
    }

    res = requests.get(url, params=params, timeout=30)
    res.raise_for_status()
    data = res.json()

    # Parse JSON time series payload
    records = []
    time_series_list = data["value"]["timeSeries"]

    for ts in time_series_list:
        param_code = ts["variable"]["variableCode"][0]["value"]
        param_name = (
            "stage_ft"
            if param_code == "00065"
            else ("discharge_cfs" if param_code == "00060" else param_code)
        )

        for val in ts["values"][0]["value"]:
            records.append(
                {
                    "site_id": site_id,
                    "datetime": val["dateTime"],
                    "value": float(val["value"]),
                    "parameter": param_name,
                }
            )

    df_ts = pd.DataFrame(records)
    if not df_ts.empty:
        # Pivot so stage_ft and discharge_cfs are separate columns
        df_pivoted = df_ts.pivot_table(
            index=["site_id", "datetime"],
            columns="parameter",
            values="value",
        ).reset_index()
        return df_pivoted

    return pd.DataFrame()

In [1]:
import os
import pandas as pd
import requests

# 1. Load CSVs (Make sure your working directory is set)
noaa_df = pd.read_csv("noaa_21-25_with_huc08.csv")
usgs_sensors_df = pd.read_csv("sensor_csv_huc08/usgs_stream_sensors_huc08.csv")

# Clean HUC8 keys
noaa_df["HUC8_clean"] = (
    pd.to_numeric(noaa_df["HUC8"], errors="coerce").fillna(-1).astype(int)
)
usgs_sensors_df["HUC8_clean"] = (
    pd.to_numeric(usgs_sensors_df["HUC8"], errors="coerce")
    .fillna(-1)
    .astype(int)
)

noaa_df["EPISODE_ID"] = noaa_df["EPISODE_ID"].astype(str)
if "NEW_EPISODE_ID" in noaa_df.columns:
    noaa_df["NEW_EPISODE_ID"] = noaa_df["NEW_EPISODE_ID"].astype(str)

# 2. Query Episode
user_ep = input("Enter EPISODE_ID to analyze: ").strip()

ep_events = noaa_df[
    (noaa_df["EPISODE_ID"] == user_ep) | (noaa_df["NEW_EPISODE_ID"] == user_ep)
]

if ep_events.empty:
    print("Episode not found!")
else:
    # Calculate Event Start & End timestamps with a 3-Hour Buffer
    ep_events["BEGIN_DT"] = pd.to_datetime(ep_events["BEGIN_DATE_TIME"])
    ep_events["END_DT"] = pd.to_datetime(ep_events["END_DATE_TIME"])

    event_start = ep_events["BEGIN_DT"].min() - pd.Timedelta(hours=3)
    event_end = ep_events["END_DT"].max() + pd.Timedelta(hours=3)

    print(f"\n==================================================")
    print(f" Episode Window: {event_start} to {event_end}")
    print(f"==================================================")

    # Find HUC8 Watersheds
    target_hucs = ep_events["HUC8_clean"].unique()
    target_hucs = [h for h in target_hucs if h != -1]

    # Find USGS Sensors in these watersheds
    matching_usgs = usgs_sensors_df[
        usgs_sensors_df["HUC8_clean"].isin(target_hucs)
    ]
    site_list = matching_usgs["foreign_id"].dropna().unique()

    print(
        f"Found {len(site_list)} USGS stream sensors in the episode watersheds."
    )

    # 3. Pull API time-series data for each sensor
    all_sensor_data = []

    for site in site_list:
        print(f"Fetching API data for USGS site {site}...")
        try:
            df_ts = get_usgs_sensor_data(site, event_start, event_end)
            if not df_ts.empty:
                all_sensor_data.append(df_ts)
        except Exception as e:
            print(f" Could not fetch data for site {site}: {e}")

    # Combine into final dataset
    if all_sensor_data:
        final_ts_df = pd.concat(all_sensor_data, ignore_index=True)
        print(
            f"\nSuccessfully downloaded {len(final_ts_df)} sensor records!"
        )

        out_file = f"episode_{user_ep}_sensor_timeseries.csv"
        final_ts_df.to_csv(out_file, index=False)
        print(f"Saved time-series data to '{out_file}'.")
        print(final_ts_df.head(10))
    else:
        print("No instantaneous sensor data returned for this time window.")

FileNotFoundError: [Errno 2] No such file or directory: 'noaa_21-25_with_huc08.csv'

In [6]:
import pandas as pd
import requests


def test_hydroiowa_api(
    sensor_numeric_id=48180, start_date="20250603", end_date="20260804"
):
    """Tests the HydroIowa REST API endpoint and handles nested JSON structures safely."""
    url = f"https://hydroiowa.org/api/riversensor/{sensor_numeric_id}/data/{start_date}/{end_date}"

    print(f"Testing HydroIowa API: {url}")

    try:
        res = requests.get(url, timeout=15)
        res.raise_for_status()

        # Parse JSON payload
        raw_json = res.json()

        # 1. Handle if raw_json is a dictionary containing a data key
        if isinstance(raw_json, dict):
            # Check common keys used by web APIs
            if "data" in raw_json:
                df = pd.DataFrame(raw_json["data"])
            elif "records" in raw_json:
                df = pd.DataFrame(raw_json["records"])
            else:
                # Normalize nested dictionary structures safely
                df = pd.json_normalize(raw_json)

        # 2. Handle if raw_json is already a list of dictionaries
        elif isinstance(raw_json, list):
            df = pd.DataFrame(raw_json)
        else:
            print("  ⚠️ Unexpected JSON structure received.")
            return False

        print(f"  ✅ SUCCESS! Received {len(df)} records.")
        print(f"     Columns: {list(df.columns)}")
        print(
            f"\n     Latest Sample Records:\n{df.tail(3).to_string(index=False)}"
        )
        return True

    except Exception as e:
        print(f"  ❌ FAILED: {e}\n")
        return False


if __name__ == "__main__":
    test_hydroiowa_api(48180, "20250603", "20260804")

Testing HydroIowa API: https://hydroiowa.org/api/riversensor/48180/data/20250603/20260804
  ✅ SUCCESS! Received 1 records.
     Columns: ['issuedTime', 'observatoryId', 'observatoryName', 'gaugeElevation', 'observed', 'parameters.waterElev.longName', 'parameters.waterElev.unit', 'parameters.discharge.longName', 'parameters.discharge.unit']

     Latest Sample Records:
         issuedTime  observatoryId observatoryName  gaugeElevation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [18]:
import io
import os
import zipfile
from datetime import datetime, timedelta
import pandas as pd
import requests

# ==========================================
# CONFIGURATION & FILE PATHS
# ==========================================
NOAA_EVENTS_FILE = "noaa_with_huc08.csv"
IFC_SENSORS_FILE = "sensor_csv_huc08/ifis_stream_sensors_huc08.csv"
IFC_HYDROSTATIONS_FILE = "sensor_csv_huc08/ifis_hydrostations_huc08.csv"
USGS_SENSORS_FILE = "sensor_csv_huc08/usgs_stream_sensors_huc08.csv"


# ==========================================
# HELPER FUNCTIONS
# ==========================================
def normalize_huc(series):
    """Normalizes HUC identifiers to 8-digit strings while preserving index alignment."""
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(8)
    )


def load_and_aggregate_episodes(filepath):
    """Loads raw NOAA event data and aggregates event rows by NEW_EPISODE_ID in memory."""
    if not os.path.exists(filepath):
        print(f"❌ Error: Required file '{filepath}' not found.")
        return None

    raw_df = pd.read_csv(filepath)

    raw_df["NEW_EPISODE_ID"] = raw_df["NEW_EPISODE_ID"].astype(str).str.strip()
    raw_df["HUC8_clean"] = normalize_huc(raw_df["HUC8"])

    episodes_df = (
        raw_df.groupby("NEW_EPISODE_ID")
        .agg(
            start_time=("MACRO_START", "min"),
            end_time=("CAPPED_EPISODE_END", "max"),
            huc08=(
                "HUC8_clean",
                lambda x: ";".join(
                    sorted(set(str(h) for h in x if str(h) != "00000000"))
                ),
            ),
        )
        .reset_index()
        .rename(columns={"NEW_EPISODE_ID": "episode_id"})
    )

    return episodes_df


# ==========================================
# API DATA FETCHERS
# ==========================================
def fetch_ifc_sensor_data(sensor_row, start_dt, end_dt):
    """
    Fetches stage time-series for an IFC River Sensor using HydroIowa API.
    Tries numeric `id` first, then falls back to `foreign_id` string code if empty.
    """
    start_str = start_dt.strftime("%Y%m%d")
    end_str = end_dt.strftime("%Y%m%d")

    num_id = (
        int(sensor_row["id"]) if pd.notna(sensor_row.get("id")) else None
    )
    code_id = sensor_row.get("foreign_id")

    candidates = [c for c in [num_id, code_id] if c is not None]

    for candidate in candidates:
        url = f"https://hydroiowa.org/api/riversensor/{candidate}/data/{start_str}/{end_str}"
        try:
            res = requests.get(url, timeout=15)
            if res.status_code == 200:
                raw_json = res.json()
                observed = raw_json.get("observed", [])
                if observed:
                    df = pd.DataFrame(observed)
                    if "validTime" in df.columns:
                        df["validTime"] = pd.to_datetime(
                            df["validTime"], format="mixed", errors="coerce"
                        )
                        df = df.dropna(subset=["validTime"])

                        if df["validTime"].dt.tz is not None:
                            df["validTime"] = df["validTime"].dt.tz_localize(
                                None
                            )

                        df = df[
                            (df["validTime"] >= start_dt)
                            & (df["validTime"] <= end_dt)
                        ].copy()

                    if not df.empty:
                        print(
                            f"      ✅ Retrieved {len(df)} records using identifier: {candidate}"
                        )
                        return df
        except Exception:
            continue

    print(
        f"      ℹ️ No stage data returned for sensor {sensor_row.get('foreign_id')}"
    )
    return pd.DataFrame()


def fetch_ifc_hydrostation_data(station_row, start_dt, end_dt):
    """
    Fetches multi-parameter data for an IFC Hydrostation via HydroIowa & IFIS APIs.
    Parses candidates from `id`, `foreign_id`, and `description`.
    """
    start_str = start_dt.strftime("%Y%m%d")
    end_str = end_dt.strftime("%Y%m%d")

    candidates = []

    # 1. Numeric ID
    if pd.notna(station_row.get("id")):
        try:
            candidates.append(int(float(station_row["id"])))
        except ValueError:
            pass

    # 2. Foreign ID
    code_id = station_row.get("foreign_id")
    if pd.notna(code_id) and str(code_id).strip():
        candidates.append(str(code_id).strip())

    # 3. Clean Name Extracted from `description` (e.g. "UPPRIOWA02 IFC-Hydrostation" -> "UPPRIOWA02")
    desc = station_row.get("description")
    if pd.notna(desc) and str(desc).strip():
        desc_clean = (
            str(desc)
            .replace("IFC-Hydrostation", "")
            .replace("Hydrostation", "")
            .strip()
            .split()[0]
        )
        if desc_clean and desc_clean not in candidates:
            candidates.append(desc_clean)

    params_to_try = ["rain", "wind", "soil", "well", "groundwell"]
    base_urls = [
        "https://hydroiowa.org/api/hydrostation",
        "https://ifis.iowafloodcenter.org/ifis/ws/data/hydrostation",
    ]

    for candidate in candidates:
        all_frames = []

        for base_url in base_urls:
            # Attempt A: Query per-parameter
            for param in params_to_try:
                url = f"{base_url}/{candidate}/data/{start_str}/{end_str}/?{param}"
                try:
                    res = requests.get(url, timeout=10)
                    if res.status_code == 200:
                        raw_json = res.json()

                        observations = []
                        if isinstance(raw_json, dict):
                            observations = raw_json.get(
                                param, raw_json.get("observed", [])
                            )
                        elif isinstance(raw_json, list):
                            observations = raw_json

                        if (
                            isinstance(observations, list)
                            and len(observations) > 0
                        ):
                            param_df = pd.DataFrame(observations)
                            if "validTime" in param_df.columns:
                                param_df["parameter_type"] = param
                                all_frames.append(param_df)
                except Exception:
                    continue

            # Attempt B: Direct time-range query
            if not all_frames:
                url = f"{base_url}/{candidate}/data/{start_str}/{end_str}"
                try:
                    res = requests.get(url, timeout=10)
                    if res.status_code == 200:
                        raw_json = res.json()
                        if isinstance(raw_json, dict):
                            for k, v in raw_json.items():
                                if isinstance(v, list) and len(v) > 0:
                                    p_df = pd.DataFrame(v)
                                    if "validTime" in p_df.columns:
                                        p_df["parameter_type"] = k
                                        all_frames.append(p_df)
                        elif isinstance(raw_json, list) and len(raw_json) > 0:
                            p_df = pd.DataFrame(raw_json)
                            if "validTime" in p_df.columns:
                                all_frames.append(p_df)
                except Exception:
                    pass

            if all_frames:
                break

        if all_frames:
            df = pd.concat(all_frames, ignore_index=True)
            if "validTime" in df.columns:
                df["validTime"] = pd.to_datetime(
                    df["validTime"], format="mixed", errors="coerce"
                )
                df = df.dropna(subset=["validTime"])

                if df["validTime"].dt.tz is not None:
                    df["validTime"] = df["validTime"].dt.tz_localize(None)

                df = df[
                    (df["validTime"] >= start_dt) & (df["validTime"] <= end_dt)
                ].copy()

            if not df.empty:
                print(
                    f"      ✅ Retrieved {len(df)} records using identifier: {candidate}"
                )
                return df

    print(
        f"      ℹ️ No hydrostation data returned for station {station_row.get('foreign_id', station_row.get('description'))}"
    )
    return pd.DataFrame()


def fetch_usgs_sensor_data(site_id, start_dt, end_dt):
    """Fetches stage time-series for a USGS Stream Sensor via NWIS IV API."""
    clean_site = str(site_id).strip()

    if not clean_site.replace(".0", "").isdigit():
        print(
            f"      ℹ️ Skipping non-numeric USGS identifier: {clean_site}"
        )
        return pd.DataFrame()

    url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {
        "format": "json",
        "sites": clean_site.zfill(8),
        "parameterCd": "00065",  # Stage only (Gage height, feet)
        "startDT": start_dt.isoformat(),
        "endDT": end_dt.isoformat(),
    }

    try:
        res = requests.get(url, params=params, timeout=15)
        res.raise_for_status()
        data = res.json()

        time_series_list = data.get("value", {}).get("timeSeries", [])
        records = []

        for ts in time_series_list:
            param_name = ts["variable"]["variableName"]
            unit = ts["variable"]["unit"]["unitCode"]
            values = ts["values"][0]["value"]

            for val in values:
                records.append(
                    {
                        "datetime": val["dateTime"],
                        "parameter": param_name,
                        "value": val["value"],
                        "unit": unit,
                    }
                )

        if not records:
            print(f"      ℹ️ No USGS stage data returned for site {clean_site}")
            return pd.DataFrame()

        df = pd.DataFrame(records)
        print(f"      ✅ Retrieved {len(df)} stage records")
        return df
    except Exception as e:
        print(f"      ❌ Exception fetching USGS Sensor {clean_site}: {e}")
        return pd.DataFrame()


# ==========================================
# MAIN PIPELINE LOGIC
# ==========================================
def main():
    print("=" * 60)
    print("      HYDROLOGICAL EVENT EPISODE DATA RETRIEVAL PIPELINE      ")
    print("=" * 60)

    episodes_df = load_and_aggregate_episodes(NOAA_EVENTS_FILE)
    if episodes_df is None:
        return

    target_episode = (
        input("\nEnter New Episode ID to search (e.g. 157601_0): ")
        .strip()
        .lower()
    )

    ep_match = episodes_df[
        episodes_df["episode_id"].str.lower() == target_episode
    ]

    if ep_match.empty:
        print(
            f"❌ New Episode ID '{target_episode}' not found in {NOAA_EVENTS_FILE}."
        )
        return

    ep_row = ep_match.iloc[0]

    start_time = pd.to_datetime(ep_row["start_time"]) - timedelta(hours=3)
    end_time = pd.to_datetime(ep_row["end_time"]) + timedelta(hours=3)

    affected_hucs = [h.strip() for h in str(ep_row["huc08"]).split(";")]

    print(f"\n📌 Episode Summary [{ep_row['episode_id']}]:")
    print(f"   • Affected Watersheds (HUC08): {', '.join(affected_hucs)}")
    print(
        f"   • Event Window (with ±3hr buffer): {start_time} to {end_time}\n"
    )

    ifc_sensors = pd.read_csv(IFC_SENSORS_FILE)
    ifc_hydrostations = pd.read_csv(IFC_HYDROSTATIONS_FILE)
    usgs_sensors = pd.read_csv(USGS_SENSORS_FILE)

    matched_ifc_sensors = ifc_sensors[
        normalize_huc(ifc_sensors["HUC8"]).isin(affected_hucs)
    ]
    matched_ifc_stations = ifc_hydrostations[
        normalize_huc(ifc_hydrostations["HUC8"]).isin(affected_hucs)
    ]
    matched_usgs_sensors = usgs_sensors[
        normalize_huc(usgs_sensors["HUC8"]).isin(affected_hucs)
    ]

    print(
        f"🔍 Sensors Found in Watershed(s):\n"
        f"   • IFC River Sensors:  {len(matched_ifc_sensors)}\n"
        f"   • IFC Hydrostations:  {len(matched_ifc_stations)}\n"
        f"   • USGS Sensors:       {len(matched_usgs_sensors)}\n"
    )

    total_sensors = (
        len(matched_ifc_sensors)
        + len(matched_ifc_stations)
        + len(matched_usgs_sensors)
    )
    if total_sensors == 0:
        print("No sensors available for this episode's watershed(s).")
        return

    confirm = (
        input("Would you like to download time-series data for these sensors? (y/n): ")
        .strip()
        .lower()
    )

    if confirm != "y":
        print("Download canceled.")
        return

    zip_filename = f"episode_{ep_row['episode_id']}_data.zip"
    print(f"\n📦 Downloading sensor data and creating '{zip_filename}'...")

    with zipfile.ZipFile(
        zip_filename, "w", zipfile.ZIP_DEFLATED
    ) as zip_archive:

        # --- A. Download IFC River Sensors ---
        print("\n[1/3] Downloading IFC River Sensors...")
        for _, row in matched_ifc_sensors.iterrows():
            sensor_code = str(
                row.get("foreign_id", row.get("id", "sensor"))
            )
            print(f"   -> Fetching IFC Sensor: {sensor_code}...")

            df = fetch_ifc_sensor_data(row, start_time, end_time)
            if not df.empty:
                zip_archive.writestr(
                    f"ifc_sensors/{sensor_code}.csv",
                    df.to_csv(index=False).encode("utf-8"),
                )

        # --- B. Download IFC Hydrostations ---
        print("\n[2/3] Downloading IFC Hydrostations...")
        for _, row in matched_ifc_stations.iterrows():
            station_code = str(
                row.get(
                    "foreign_id",
                    row.get("description", row.get("id", "station")),
                )
            ).replace(" ", "_")
            print(f"   -> Fetching Hydrostation: {station_code}...")

            df = fetch_ifc_hydrostation_data(row, start_time, end_time)
            if not df.empty:
                zip_archive.writestr(
                    f"ifc_hydrostations/{station_code}.csv",
                    df.to_csv(index=False).encode("utf-8"),
                )

        # --- C. Download USGS Sensors ---
        print("\n[3/3] Downloading USGS Sensors...")
        for _, row in matched_usgs_sensors.iterrows():
            site_id = str(row.get("foreign_id", row.get("id"))).zfill(8)
            print(f"   -> Fetching USGS Sensor: {site_id}...")

            df = fetch_usgs_sensor_data(site_id, start_time, end_time)
            if not df.empty:
                zip_archive.writestr(
                    f"usgs_sensors/{site_id}.csv",
                    df.to_csv(index=False).encode("utf-8"),
                )

    print("\n==========================================")
    print(f"✨ PIPELINE COMPLETE!")
    print(f"All time-series files saved to: {zip_filename}")
    print("==========================================")


if __name__ == "__main__":
    main()

      HYDROLOGICAL EVENT EPISODE DATA RETRIEVAL PIPELINE      



📌 Episode Summary [191899_0]:
   • Affected Watersheds (HUC08): 07100004, 07100006, 10170203, 10170204, 10230002, 10230003, 10230005
   • Event Window (with ±3hr buffer): 2024-06-20 12:45:00 to 2024-06-23 18:45:00

🔍 Sensors Found in Watershed(s):
   • IFC River Sensors:  38
   • IFC Hydrostations:  4
   • USGS Sensors:       57


📦 Downloading sensor data and creating 'episode_191899_0_data.zip'...

[1/3] Downloading IFC River Sensors...
   -> Fetching IFC Sensor: WATERMAN01...
      ✅ Retrieved 238 records using identifier: WATERMAN01
   -> Fetching IFC Sensor: DSMNSRV03...
      ✅ Retrieved 238 records using identifier: DSMNSRV03
   -> Fetching IFC Sensor: WILLOWCR01...
      ✅ Retrieved 238 records using identifier: WILLOWCR01
   -> Fetching IFC Sensor: NRCCNRV03...
      ℹ️ No stage data returned for sensor NRCCNRV03
   -> Fetching IFC Sensor: WLNTCR02...
      ✅ Retrieved 238 records using identifier: WLNTCR02
   -> Fetching IFC Sensor: LTLSIOUX02...
      ✅ Retrieved 238 record

In [21]:
from datetime import datetime
import pandas as pd
import requests


def fetch_ifc_hydrostation_data(station_row, start_dt, end_dt):
    """Fetches multi-parameter data for an IFC Hydrostation via HydroIowa API."""
    start_str = start_dt.strftime("%Y%m%d")
    end_str = end_dt.strftime("%Y%m%d")

    numeric_id = None
    if pd.notna(station_row.get("id")):
        try:
            numeric_id = str(int(float(station_row["id"])))
        except ValueError:
            pass

    if not numeric_id:
        print(
            f"❌ Skipping station {station_row.get('foreign_id')}: No valid numeric ID found."
        )
        return pd.DataFrame()

    params_to_try = ["rain", "wind", "soil", "well", "groundwell"]
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    all_frames = []

    for param in params_to_try:
        url = f"https://hydroiowa.org/api/hydrostation/{numeric_id}/data/{start_str}/{end_str}/?{param}"
        try:
            res = requests.get(url, headers=headers, timeout=10)
            if res.status_code == 200:
                raw_json = res.json()

                observed_dict = raw_json.get("observed", {})
                param_data = observed_dict.get(param, [])

                if isinstance(param_data, list) and len(param_data) > 0:
                    param_df = pd.DataFrame(param_data)
                    param_df["parameter_type"] = param
                    all_frames.append(param_df)
        except Exception:
            continue

    if not all_frames:
        print(
            f"ℹ️ No hydrostation data returned for ID {numeric_id} ({station_row.get('foreign_id')})"
        )
        return pd.DataFrame()

    df = pd.concat(all_frames, ignore_index=True)

    if "validTime" in df.columns:
        df["validTime"] = pd.to_datetime(
            df["validTime"], format="mixed", errors="coerce"
        )
        df = df.dropna(subset=["validTime"])

        if df["validTime"].dt.tz is not None:
            df["validTime"] = df["validTime"].dt.tz_localize(None)

        df = df[(df["validTime"] >= start_dt) & (df["validTime"] <= end_dt)]

    print(
        f"✅ Success! Retrieved {len(df)} total records for Hydrostation {numeric_id} ({station_row.get('foreign_id')})"
    )
    return df


# --- VERIFICATION RUN ---
if __name__ == "__main__":
    # Simulating a row from ifis_hydrostations_huc08.csv
    test_station_row = {
        "id": 5089,
        "foreign_id": "UPPRIOWA02",
        "description": "UPPRIOWA02 IFC-Hydrostation",
    }

    start_date = datetime(2026, 6, 1)
    end_date = datetime(2026, 6, 5)

    print("Running hydrostation data extraction verification...\n")
    df_result = fetch_ifc_hydrostation_data(
        test_station_row, start_date, end_date
    )

    if not df_result.empty:
        print("\nPreview of retrieved DataFrame:")
        print(
            df_result[
                [
                    "validTime",
                    "parameter_type",
                    "accum",
                    "precipDurration",
                ]
            ].head()
        )
        

Running hydrostation data extraction verification...

✅ Success! Retrieved 1540 total records for Hydrostation 5089 (UPPRIOWA02)

Preview of retrieved DataFrame:
            validTime parameter_type  accum  precipDurration
0 2026-06-01 00:00:00           rain  0.003             60.0
1 2026-06-01 00:15:00           rain  0.003            140.0
2 2026-06-01 00:30:00           rain  0.006            290.0
3 2026-06-01 00:45:00           rain  0.009            210.0
4 2026-06-01 01:00:00           rain  0.012            410.0


In [22]:
import io
import pandas as pd
import requests

# 1. Fetch pipe-delimited text from IFIS API
url = "https://ifis.iowafloodcenter.org/ifis/ws/meta/ifis.objects.php?type=2,3"
response = requests.get(url)

if response.status_code == 200:
    raw_text = response.text

    # Clean the header row (remove leading '#')
    lines = raw_text.strip().split("\n")
    headers = [col.strip() for col in lines[0].lstrip("#").split("|")]

    # Load API data into DataFrame
    df_ifis = pd.read_csv(
        io.StringIO("\n".join(lines[1:])), sep="|", names=headers
    )

    # Convert 'id' to integer for matching
    df_ifis["id"] = pd.to_numeric(df_ifis["id"], errors="coerce")

    # 2. Load local CSV file
    input_csv = "usgs_stream_sensors_huc08_revised.csv"
    output_csv = "usgs_stream_sensors_huc08_with_foreign_id1.csv"
    df_local = pd.read_csv(input_csv)

    # 3. Merge foreign_id1 on 'id'
    df_merged = df_local.merge(
        df_ifis[["id", "foreign_id1"]], on="id", how="left"
    )

    # 4. Position foreign_id1 directly after foreign_id
    cols = df_merged.columns.tolist()
    if "foreign_id" in cols and "foreign_id1" in cols:
        fid_idx = cols.index("foreign_id")
        cols.remove("foreign_id1")
        cols.insert(fid_idx + 1, "foreign_id1")
        df_merged = df_merged[cols]

    # Save output
    df_merged.to_csv(output_csv, index=False)
    print(
        f"Successfully merged foreign_id1 from pipe-delimited IFIS feed to {output_csv}"
    )
else:
    print(f"Failed to fetch data. HTTP Status: {response.status_code}")

Successfully merged foreign_id1 from pipe-delimited IFIS feed to usgs_stream_sensors_huc08_with_foreign_id1.csv


In [9]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta
import io
import os
import time
import zipfile
import pandas as pd
import requests

# ==========================================
# CONFIGURATION & FILE PATHS
# ==========================================
MAX_WORKERS = (
    6  # Number of concurrent download threads to avoid API throttling
)
MAX_RETRIES = 2  # Retries per sensor on empty/failed result
RETRY_DELAY_SECONDS = 4
NOAA_EVENTS_FILE = "noaa_21-25_with_huc_08.csv"
IFC_SENSORS_FILE = "sensor_csv_huc08/ifis_stream_sensors_huc08.csv"
IFC_HYDROSTATIONS_FILE = "sensor_csv_huc08/ifis_hydrostations_huc08.csv"
USGS_SENSORS_FILE = "sensor_csv_huc08/usgs_stream_sensors_huc08_with_foreign_id1.csv"


# ==========================================
# HELPER FUNCTIONS
# ==========================================
def normalize_huc(series):
    """Normalizes HUC identifiers to 8-digit strings while preserving index alignment."""
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
        .str.zfill(8)
    )


def load_and_aggregate_episodes(filepath):
    """Loads raw NOAA event data and aggregates event rows by NEW_EPISODE_ID in memory."""
    if not os.path.exists(filepath):
        print(f"❌ Error: Required file '{filepath}' not found.")
        return None

    raw_df = pd.read_csv(filepath)

    raw_df["NEW_EPISODE_ID"] = raw_df["NEW_EPISODE_ID"].astype(str).str.strip()
    raw_df["HUC8_clean"] = normalize_huc(raw_df["HUC8"])

    episodes_df = (
        raw_df.groupby("NEW_EPISODE_ID")
        .agg(
            start_time=("MACRO_START", "min"),
            end_time=("CAPPED_EPISODE_END", "max"),
            huc08=(
                "HUC8_clean",
                lambda x: ";".join(
                    sorted(set(str(h) for h in x if str(h) != "00000000"))
                ),
            ),
        )
        .reset_index()
        .rename(columns={"NEW_EPISODE_ID": "episode_id"})
    )

    return episodes_df


def resolve_hydrostation_id(station_row):
    """Resolves the true numeric IFC ID directly from foreign_id1, falling back to other columns if necessary."""
    foreign_id1 = station_row.get("foreign_id1")
    if pd.notna(foreign_id1):
        try:
            return str(int(float(foreign_id1)))
        except ValueError:
            pass

    for col in [
        "ifc_id",
        "station_id",
        "hydro_id",
        "ifis_id",
        "ifc_id.1",
        "Sensor_ID",
    ]:
        val = station_row.get(col)
        if pd.notna(val) and str(val).strip() != "":
            try:
                num = int(float(val))
                if num > 4000:
                    return str(num)
            except ValueError:
                pass

    raw_id = station_row.get("id")
    if pd.notna(raw_id):
        try:
            return str(int(float(raw_id)))
        except ValueError:
            pass

    return None


def fetch_nws_lid_data(nws_id, start_dt, end_dt):
    """Fallback fetcher: Queries NOAA's National Water Prediction Service (NWPS) API

    using NWS Location Identifiers (LIDs) like CBSS2, MRLI4, etc.
    """
    nws_id = str(nws_id).strip().upper()
    url = f"https://api.water.noaa.gov/v1/gauges/{nws_id}/stageflow/observed"

    headers = {
        "User-Agent": "Mozilla/5.0 HydrologicalPipeline/2.0",
        "Accept": "application/json",
    }

    try:
        res = requests.get(url, headers=headers, timeout=12)
        if res.status_code == 200:
            data = res.json().get("data", [])
            records = []
            for obs in data:
                raw_time = obs.get("validTime")
                if not raw_time:
                    continue

                valid_time = pd.to_datetime(raw_time)
                if valid_time.tzinfo is not None:
                    valid_time = valid_time.tz_localize(None)

                if start_dt <= valid_time <= end_dt:
                    records.append(
                        {
                            "datetime": valid_time,
                            "parameter": "Stage / Observed",
                            "value": obs.get("primary"),
                            "unit": obs.get("primaryUnit", "ft"),
                        }
                    )

            if records:
                df = pd.DataFrame(records).sort_values("datetime")
                print(
                    f"      ✅ Retrieved {len(df)} records for NWS LID {nws_id} (via NOAA NWPS API)"
                )
                return df
    except Exception:
        pass

    return pd.DataFrame()


# ==========================================
# API DATA FETCHERS
# ==========================================
def fetch_ifc_sensor_data(sensor_row, start_dt, end_dt):
    """Fetches stage time-series for an IFC River Sensor using HydroIowa API."""
    start_str = start_dt.strftime("%Y%m%d")
    end_str = end_dt.strftime("%Y%m%d")

    num_id = (
        int(sensor_row["id"]) if pd.notna(sensor_row.get("id")) else None
    )
    code_id = sensor_row.get("foreign_id")

    candidates = [c for c in [num_id, code_id] if c is not None]

    for candidate in candidates:
        url = f"https://hydroiowa.org/api/riversensor/{candidate}/data/{start_str}/{end_str}"
        try:
            res = requests.get(url, timeout=15)
            if res.status_code == 200:
                raw_json = res.json()
                observed = raw_json.get("observed", [])
                if observed:
                    df = pd.DataFrame(observed)
                    if "validTime" in df.columns:
                        df["validTime"] = pd.to_datetime(
                            df["validTime"], format="mixed", errors="coerce"
                        )
                        df = df.dropna(subset=["validTime"])

                        if df["validTime"].dt.tz is not None:
                            df["validTime"] = df["validTime"].dt.tz_localize(
                                None
                            )

                        df = df[
                            (df["validTime"] >= start_dt)
                            & (df["validTime"] <= end_dt)
                        ].copy()

                    if not df.empty:
                        print(
                            f"      ✅ Retrieved {len(df)} stage records using identifier: {candidate}"
                        )
                        return df
        except Exception:
            continue

    print(
        f"      ℹ️ No stage data returned for sensor {sensor_row.get('foreign_id')}"
    )
    return pd.DataFrame()


def fetch_ifc_hydrostation_data(station_row, start_dt, end_dt):
    """Fetches multi-parameter data for an IFC Hydrostation via HydroIowa API."""
    start_str = start_dt.strftime("%Y%m%d")
    end_str = end_dt.strftime("%Y%m%d")

    numeric_id = resolve_hydrostation_id(station_row)
    station_name = station_row.get("foreign_id1")

    if not numeric_id:
        print(
            f"      ❌ Skipping station {station_name}: No valid numeric ID found."
        )
        return pd.DataFrame()

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    all_frames = []
    params_to_try = ["rain", "wind", "soil", "well", "groundwell", "stage"]

    # Attempt 1: Combined Parameter Query
    query_flags = "&".join(params_to_try)
    url = f"https://hydroiowa.org/api/hydrostation/{numeric_id}/data/{start_str}/{end_str}/?{query_flags}"
    try:
        res = requests.get(url, headers=headers, timeout=15)
        if res.status_code == 200:
            raw_json = res.json()
            observed_dict = raw_json.get("observed", {})

            if isinstance(observed_dict, dict):
                for param in params_to_try:
                    param_data = observed_dict.get(param, [])
                    if isinstance(param_data, list) and len(param_data) > 0:
                        param_df = pd.DataFrame(param_data)
                        param_df["parameter_type"] = param
                        all_frames.append(param_df)
            elif isinstance(observed_dict, list) and len(observed_dict) > 0:
                param_df = pd.DataFrame(observed_dict)
                all_frames.append(param_df)
    except Exception:
        pass

    # Attempt 2: Direct /data Call
    if not all_frames:
        url = f"https://hydroiowa.org/api/hydrostation/{numeric_id}/data/{start_str}/{end_str}"
        try:
            res = requests.get(url, headers=headers, timeout=10)
            if res.status_code == 200:
                raw_json = res.json()
                observed_dict = raw_json.get("observed", {})

                if isinstance(observed_dict, dict):
                    for k, v in observed_dict.items():
                        if isinstance(v, list) and len(v) > 0:
                            p_df = pd.DataFrame(v)
                            p_df["parameter_type"] = k
                            all_frames.append(p_df)
                elif isinstance(observed_dict, list) and len(observed_dict) > 0:
                    p_df = pd.DataFrame(observed_dict)
                    all_frames.append(p_df)
        except Exception:
            pass

    if not all_frames:
        print(
            f"      ℹ️ No hydrostation data returned for ID {numeric_id} ({station_name}) during this event window."
        )
        return pd.DataFrame()

    df = pd.concat(all_frames, ignore_index=True)

    if "validTime" in df.columns:
        df["validTime"] = pd.to_datetime(
            df["validTime"], format="mixed", errors="coerce"
        )
        df = df.dropna(subset=["validTime"])

        if df["validTime"].dt.tz is not None:
            df["validTime"] = df["validTime"].dt.tz_localize(None)

        df = df[(df["validTime"] >= start_dt) & (df["validTime"] <= end_dt)]

    if (
        "validTime" in df.columns
        and "parameter_type" in df.columns
        and not df.empty
    ):
        merge_cols = [
            c
            for c in df.columns
            if c not in ("validTime", "parameter_type")
        ]
        df = df.groupby("validTime", as_index=False)[merge_cols].first()
        df = df.sort_values("validTime")

    if not df.empty:
        print(
            f"      ✅ Retrieved {len(df)} records for Hydrostation {numeric_id} ({station_name})"
        )
    else:
        print(
            f"      ℹ️ No hydrostation records in range for ID {numeric_id} ({station_name})"
        )

    return df


def fetch_usgs_sensor_data(usgs_row, start_dt, end_dt):
    """Fetches Instantaneous Values (IV) time-series data for a USGS / NWS Stream Sensor.

    1. Scans candidate ID columns (foreign_id1, foreign_id, usgs_site_no_revised, id).
    2. Queries USGS NWIS IV service using numeric 'sites' and uppercase 'nwsLids'.
    3. Falls back to NOAA's NWPS API if USGS NWIS returns no records for NWS LIDs.
    """
    candidate_keys = [
        "foreign_id1",
        "foreign_id",
        "usgs_site_no_revised",
        "usgs_site_no",
        "id",
    ]

    candidates = []
    nws_lids = []

    for key in candidate_keys:
        val = usgs_row.get(key)
        if pd.notna(val):
            clean_val = str(val).split(".")[0].strip().upper()
            if clean_val and clean_val != "NAN" and clean_val not in candidates:
                candidates.append(clean_val)
                if not clean_val.isdigit():
                    nws_lids.append(clean_val)

    if not candidates:
        print("      ❌ Skipping USGS row: No valid station identifier found.")
        return pd.DataFrame()

    # Step 1: Attempt USGS NWIS Queries
    strategies = []
    for site_id in candidates:
        if site_id.isdigit():
            padded_id = site_id.zfill(8)
            strategies.append(("sites", padded_id))
            strategies.append(("nwsLids", site_id))
        else:
            strategies.append(("nwsLids", site_id))
            strategies.append(("sites", site_id))

    url = "https://waterservices.usgs.gov/nwis/iv/"
    parameter_attempts = [
        "00065",
        "00060",
        None,
    ]  # 00065 = Stage/Gage height, 00060 = Discharge

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) HydrologicalPipeline/2.0"
    }

    for param_type, query_val in strategies:
        for param_cd in parameter_attempts:
            params = {
                "format": "json",
                param_type: query_val,
                "startDT": start_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
                "endDT": end_dt.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
            }
            if param_cd:
                params["parameterCd"] = param_cd

            try:
                res = requests.get(
                    url, params=params, headers=headers, timeout=15
                )
                if res.status_code != 200:
                    continue

                data = res.json()
                time_series_list = (
                    data.get("value", {}).get("timeSeries", [])
                )
                records = []

                for ts in time_series_list:
                    param_name = ts.get("variable", {}).get(
                        "variableName", "Unknown"
                    )
                    unit = (
                        ts.get("variable", {})
                        .get("unit", {})
                        .get("unitCode", "N/A")
                    )
                    values = ts.get("values", [{}])[0].get("value", [])

                    for val in values:
                        records.append(
                            {
                                "datetime": val.get("dateTime"),
                                "parameter": param_name,
                                "value": val.get("value"),
                                "unit": unit,
                            }
                        )

                if records:
                    df = pd.DataFrame(records)
                    print(
                        f"      ✅ Retrieved {len(df)} records for USGS {query_val} (via {param_type})"
                    )
                    return df

            except Exception:
                continue

    # Step 2: Fallback to NOAA NWPS API for NWS LIDs (e.g. CBSS2, MRLI4, WWDI4)
    for lid in nws_lids:
        df_noaa = fetch_nws_lid_data(lid, start_dt, end_dt)
        if df_noaa is not None and not df_noaa.empty:
            return df_noaa

    primary_id = candidates[0]
    print(
        f"      ℹ️ No stage/discharge data available across USGS/NOAA APIs for site {primary_id}"
    )
    return pd.DataFrame()


def fetch_with_retry(fetch_fn, fetch_args):
    """Calls fetch_fn(*fetch_args), retrying on empty result or exception."""
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            df = fetch_fn(*fetch_args)
            if df is not None and not df.empty:
                return df
        except Exception as exc:
            last_exc = exc

        if attempt < MAX_RETRIES:
            time.sleep(RETRY_DELAY_SECONDS)

    if last_exc:
        raise last_exc
    return pd.DataFrame()


# ==========================================
# MAIN PIPELINE LOGIC
# ==========================================
def main():
    print("=" * 60)
    print("      HYDROLOGICAL EVENT EPISODE DATA RETRIEVAL PIPELINE      ")
    print("=" * 60)

    episodes_df = load_and_aggregate_episodes(NOAA_EVENTS_FILE)
    if episodes_df is None:
        return

    target_episode = (
        input("\nEnter New Episode ID to search (e.g. 157601_0): ")
        .strip()
        .lower()
    )

    ep_match = episodes_df[
        episodes_df["episode_id"].str.lower() == target_episode
    ]

    if ep_match.empty:
        print(
            f"❌ New Episode ID '{target_episode}' not found in {NOAA_EVENTS_FILE}."
        )
        return

    ep_row = ep_match.iloc[0]

    start_time = pd.to_datetime(ep_row["start_time"]) - timedelta(hours=3)
    end_time = pd.to_datetime(ep_row["end_time"]) + timedelta(hours=3)

    affected_hucs = [h.strip() for h in str(ep_row["huc08"]).split(";")]

    print(f"\n📌 Episode Summary [{ep_row['episode_id']}]:")
    print(f"   • Affected Watersheds (HUC08): {', '.join(affected_hucs)}")
    print(
        f"   • Event Window (with ±3hr buffer): {start_time} to {end_time}\n"
    )

    ifc_sensors = pd.read_csv(IFC_SENSORS_FILE)
    ifc_hydrostations = pd.read_csv(IFC_HYDROSTATIONS_FILE)
    usgs_sensors = pd.read_csv(USGS_SENSORS_FILE)

    matched_ifc_sensors = ifc_sensors[
        normalize_huc(ifc_sensors["HUC8"]).isin(affected_hucs)
    ]
    matched_ifc_stations = ifc_hydrostations[
        normalize_huc(ifc_hydrostations["HUC8"]).isin(affected_hucs)
    ]
    matched_usgs_sensors = usgs_sensors[
        normalize_huc(usgs_sensors["HUC8"]).isin(affected_hucs)
    ]

    print(
        f"🔍 Sensors Found in Watershed(s):\n"
        f"   • IFC River Sensors:  {len(matched_ifc_sensors)}\n"
        f"   • IFC Hydrostations:  {len(matched_ifc_stations)}\n"
        f"   • USGS Sensors:       {len(matched_usgs_sensors)}\n"
    )

    total_sensors = (
        len(matched_ifc_sensors)
        + len(matched_ifc_stations)
        + len(matched_usgs_sensors)
    )
    if total_sensors == 0:
        print("No sensors available for this episode's watershed(s).")
        return

    confirm = (
        input(
            "Would you like to download time-series data for these sensors?"
            " (y/n): "
        )
        .strip()
        .lower()
    )

    if confirm != "y":
        print("Download canceled.")
        return

    zip_filename = f"episode_{ep_row['episode_id']}_data.zip"
    print(
        f"\n📦 Downloading sensor data ({total_sensors} sensors, "
        f"{MAX_WORKERS} at a time) and creating '{zip_filename}'...\n"
    )

    tasks = []

    for _, row in matched_ifc_sensors.iterrows():
        sensor_code = str(row.get("foreign_id1", row.get("id", "sensor")))
        tasks.append(
            (
                "ifc_sensors",
                sensor_code,
                fetch_ifc_sensor_data,
                (row, start_time, end_time),
            )
        )

    for _, row in matched_ifc_stations.iterrows():
        station_code = str(
            row.get("foreign_id1", row.get("id", "station"))
        ).replace(" ", "_")
        tasks.append(
            (
                "ifc_hydrostations",
                station_code,
                fetch_ifc_hydrostation_data,
                (row, start_time, end_time),
            )
        )

    for _, row in matched_usgs_sensors.iterrows():
        site_code = str(
            row.get(
                "foreign_id1",
                row.get(
                    "usgs_site_no_revised",
                    row.get("foreign_id", row.get("id", "usgs")),
                ),
            )
        ).split(".")[0].strip().upper()
        tasks.append(
            (
                "usgs_sensors",
                site_code,
                fetch_usgs_sensor_data,
                (row, start_time, end_time),
            )
        )

    completed = 0
    success_counts = {
        "ifc_sensors": 0,
        "ifc_hydrostations": 0,
        "usgs_sensors": 0,
    }
    total_counts = {
        "ifc_sensors": 0,
        "ifc_hydrostations": 0,
        "usgs_sensors": 0,
    }
    for folder, _code, _fetch_fn, _fetch_args in tasks:
        total_counts[folder] += 1

    with zipfile.ZipFile(
        zip_filename, "w", zipfile.ZIP_DEFLATED
    ) as zip_archive:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_task = {
                executor.submit(
                    fetch_with_retry, fetch_fn, fetch_args
                ): (folder, code)
                for folder, code, fetch_fn, fetch_args in tasks
            }

            for future in as_completed(future_to_task):
                folder, code = future_to_task[future]
                completed += 1
                try:
                    df = future.result()
                except Exception as exc:
                    print(f"      ⚠️ Error fetching {folder}/{code}: {exc}")
                    continue

                if df is not None and not df.empty:
                    success_counts[folder] += 1
                    zip_archive.writestr(
                        f"{folder}/{code}.csv",
                        df.to_csv(index=False).encode("utf-8"),
                    )

                if completed % 10 == 0 or completed == len(tasks):
                    print(
                        f"   ...{completed}/{len(tasks)} sensors processed"
                    )

    print("\n==========================================")
    print("PIPELINE COMPLETE!")
    print(f"All time-series files saved to: {zip_filename}")
    print()
    print("📊 Collection Summary:")
    print(
        f"   • IFC River Sensors:  {success_counts['ifc_sensors']}/{total_counts['ifc_sensors']}"
    )
    print(
        f"   • IFC Hydrostations:  {success_counts['ifc_hydrostations']}/{total_counts['ifc_hydrostations']}"
    )
    print(
        f"   • USGS Sensors:       {success_counts['usgs_sensors']}/{total_counts['usgs_sensors']}"
    )
    grand_total_success = sum(success_counts.values())
    grand_total = sum(total_counts.values())
    print(f"   • TOTAL:              {grand_total_success}/{grand_total}")
    print("==========================================")


if __name__ == "__main__":
    main()

      HYDROLOGICAL EVENT EPISODE DATA RETRIEVAL PIPELINE      

📌 Episode Summary [194228_0]:
   • Affected Watersheds (HUC08): 07080106, 07080209
   • Event Window (with ±3hr buffer): 2024-07-02 14:10:00 to 2024-07-03 00:00:00

🔍 Sensors Found in Watershed(s):
   • IFC River Sensors:  15
   • IFC Hydrostations:  7
   • USGS Sensors:       17

Download canceled.
